# NLP HW 03: Statistical Language Modeling
## Submission By Sravanth Chowdary Potluri und5uv



Submission Deadline: __November 11, 2024, 11:59 PM__

A penalty will be applied for late submission. Please refer to the course policy for more detail.  

## Instructions

Please read the instructions carefully before you start working on the homework.

- Please follow instructions and printed out the results as required. Keep the printed results and your implementation for grading purpose.
    - The TAs will not run your code for grading purpose unless it is necessary. That means, you may lose some points if the printed results are not in the submitted file.
- Submission should be via Canvas.
    - If you use Google Colab for running the code, please download the file and submit it via Canvas once it's done.
    - Submission via a Google Colab link will be considered as an invalid submission.
- Please double check the submitted file once you upload it to Canvas.
    - Students should be responsible for checking whether they submit the right files.
    - Re-submission is not allowed once the deadline is passed.

Also, if you missed the class lectures, please study the course materials first before working on the homework. It may save you some time.

# Statistical Language Modeling

## Data Preparation

In [23]:
import urllib.request
from os.path import isfile

from scipy.stats import alpha

if not isfile("lang-model.txt"):
    url = "https://yangfengji.net/uva-nlp-grad/data/lang-model.txt.zip"
    print("Downloading ...")
    filename, headers = urllib.request.urlretrieve(url, filename="lang-model.txt.zip")

    print("Decompressing the file ...")
    !unzip lang-model.txt.zip

sents = open("lang-model.txt").read().split("\n")
print("Read {} sentences".format(len(sents)))

Read 71851 sentences


Import the packages that are necessary for this project.

In [24]:
from collections import defaultdict
from math import log2, pow
from numpy.random import choice
import numpy as np
from tqdm import tqdm

## Tri-gram Language Model (8 points)

The majority of the work will be in the following `TrigramLM` class.

TODO:

There are four functions that require your implementation

- `build`: (2 points) the main function of building the tri-gram model with smoothing parameter $\alpha$. Please refer to the course lecture for the smoothing techniques
- `logprob`: (2 points) calculate a list of log probabilities of a given sentence
- `_generate_random`: (2 points) the generation function with random sampling
- `_generate_top_p`: (2 points) the generation function with top-$p$ sampling

In [25]:
class TrigramLM(object):
    def __init__(self):
        self.vocab = {"<start>": 0, "<end>": 1}  # Vocabulary with pre-defined tokens
        self.bigram_counts = None
        self.trigram_counts = None
        self.V = 0
        self.alpha = 0.001  # Default smoothing parameter

    def build(self, fname, alpha=0.001):
        """ Build a Trigram LM
        Input:
            fname: string, the name of the file
            alpha: smoothing parameter
        """
        import collections
        self.alpha = alpha  # Store alpha for use in logprob
        vocab_counter = collections.Counter()
        self.trigram_counts = collections.Counter()
        self.bigram_counts = collections.Counter()

        with open(fname, 'r') as f:
            for line in f:
                tokens = line.strip().split()
                tokens = ['<start>'] + tokens
                vocab_counter.update(tokens)
                for i in range(len(tokens) - 2):
                    trigram = (tokens[i], tokens[i+1], tokens[i+2])
                    bigram = (tokens[i], tokens[i+1])
                    self.trigram_counts[trigram] += 1
                    self.bigram_counts[bigram] += 1

        # Build vocabulary
        for token in vocab_counter:
            if token not in self.vocab:
                self.vocab[token] = len(self.vocab)

        self.V = len(self.vocab)  # Vocabulary size

    def logprob(self, text):
        """ Evaluate a given text
        Input:
            text: a string
        Outputs:
            tokens: a list of tokens (excluding <start> and <end>)
            logprob: a list of log-probabilities of the tokens
        """
        import math
        tokens = text.strip().split()
        tokens = ['<start>'] + tokens  
        logprob = []
        for i in range(2, len(tokens)-1):
            prev_tok1 = tokens[i-2]
            prev_tok2 = tokens[i-1]
            curr_tok = tokens[i]
    
            if prev_tok1 not in self.vocab:
                prev_tok1 = 'UNK'
            if prev_tok2 not in self.vocab:
                prev_tok2 = 'UNK'
            if curr_tok not in self.vocab:
                curr_tok = 'UNK'
    
            bigram = (prev_tok1, prev_tok2)
            trigram = (prev_tok1, prev_tok2, curr_tok)
    
            trigram_count = self.trigram_counts.get(trigram, 0)
            bigram_count = self.bigram_counts.get(bigram, 0)
    
            if bigram_count == 0:
                # If bigram_count is zero, assign uniform probability
                prob = 1 / self.V
            else:
                # Apply add-alpha smoothing
                prob = (trigram_count + self.alpha) / (bigram_count + self.alpha * self.V)
    
            # Handle case when prob is zero to avoid log(0)
            if prob == 0:
                prob = 1/self.V # Assign uniform probability
    
            logprob.append(math.log(prob, 2)) # Log base 2
        return tokens[2:-1], logprob  # Exclude <start> and <end> tokens

    def generate(self, method, length=20, hp=None):
        """Implement two text generation methods
        Input:
            method: a string
            length: int, cut-off length
            hp: hyperparameter for the method
        Output:
            text: a string
        """
        if method == "random":
            text = self._generate_random(length, hp)
        elif method == "top-p":
            text = self._generate_top_p(length, hp)
        else:
            raise ValueError("Unknown method")
        return text
    
    def _generate_random(self, length, hp):
        # Implement the random sampling algorithm with hyper-parameter hp (temperature)
        import random
        temp = hp if hp is not None else 1.0  # Default temperature
        text = []
        prev_tokens = ['<start>', '<start>']
        for _ in range(length):
            bigram = (prev_tokens[-2], prev_tokens[-1])
            bigram_count = self.bigram_counts.get(bigram, 0)
            alpha = self.alpha
            V = self.V
            tokens = list(self.vocab.keys())
            probabilities = []
            if bigram_count == 0:
                # If bigram not seen, assign uniform probability
                prob = 1 / V
                probabilities = [prob] * V
            else:
                # Compute probabilities for each token
                for token in tokens:
                    trigram = (bigram[0], bigram[1], token)
                    trigram_count = self.trigram_counts.get(trigram, 0)
                    prob = (trigram_count + alpha) / (bigram_count + alpha * V)
                    probabilities.append(prob)
            # Apply temperature scaling
            probabilities = [max(p, 1e-10) for p in probabilities]  # Avoid zeros
            probabilities = [np.exp(np.log(p) / temp) for p in probabilities]
            total = sum(probabilities)
            probabilities = [p / total for p in probabilities]
            next_token = random.choices(tokens, probabilities)[0]
            if next_token == '<end>':
                break
            text.append(next_token)
            prev_tokens.append(next_token)
        return ' '.join(text)

    def _generate_top_p(self, length, hp):
        # Implement the top-p algorithm with hyper-parameter hp (p-value)
        import random
        p = hp if hp is not None else 0.9  # Default p value
        text = []
        prev_tokens = ['<start>', '<start>']
        for _ in range(length):
            bigram = (prev_tokens[-2], prev_tokens[-1])
            bigram_count = self.bigram_counts.get(bigram, 0)
            alpha = self.alpha
            V = self.V
            tokens = list(self.vocab.keys())
            probabilities = []
            if bigram_count == 0:
                # If bigram not seen, assign uniform probability
                prob = 1 / V
                probabilities = [prob] * V
            else:
                # Compute probabilities for each token
                for token in tokens:
                    trigram = (bigram[0], bigram[1], token)
                    trigram_count = self.trigram_counts.get(trigram, 0)
                    prob = (trigram_count + alpha) / (bigram_count + alpha * V)
                    probabilities.append(prob)
            # Sort tokens by probability
            tokens_probs = sorted(zip(tokens, probabilities), key=lambda x: x[1], reverse=True)
            # Apply cumulative probability threshold p
            cumulative_prob = 0.0
            tokens_top_p = []
            probs_top_p = []
            for token, prob in tokens_probs:
                cumulative_prob += prob
                tokens_top_p.append(token)
                probs_top_p.append(prob)
                if cumulative_prob >= p:
                    break
            # Normalize probabilities
            total_prob = sum(probs_top_p)
            probs_top_p = [prob / total_prob for prob in probs_top_p]
            next_token = random.choices(tokens_top_p, probs_top_p)[0]
            if next_token == '<end>':
                break
            text.append(next_token)
            prev_tokens.append(next_token)
        return ' '.join(text)

### Instead of generating the whole transition matrix containing the probabilities which is computationally expensive, we can generate the probabilities on-the-fly from only the bigram and trigram counts. This will save memory and computation time.

#### Also please note that we are using 2 start tokens at the beginning of each sentence to act as padding for the initial trigram. In both training and generation, we will start with these two tokens. To account for this sometimes one or two start tokens are appended to the text. This is not a mistake

## Training

The following code block will load the training file and build the tri-gram language model.

In [26]:
# build trigram at alpha=0.001
trigram = TrigramLM()
trigram.build("lang-model.txt",alpha=0.001)

In [27]:
# building the model with alpha=0.1
trigram_0_1 = TrigramLM()
trigram_0_1.build("lang-model.txt",alpha=0.1)

## Evaluation (4 points)

Please use the following code block and implement the function to calculate the perplexity.

### Perplexity Calculation (2 points)

TODO:

- Please implement the calculation of perplexity
- Please refer to the lecture slides of the definition of perplexity

In [28]:
def perplexity(model, fname):
    """ Read the texts from fname and calculate the perplexity
    Input:
        model: a TrigramLM instance
        fname: string, the name of the file
    Output:
        ppl: float, the perplexity
    """
    import math
    total_logprob = 0.0
    total_words = 0
    with open(fname, 'r') as f:
        for line in f:
            tokens, logprob = model.logprob(line.strip())
            total_logprob += sum(logprob)
            total_words += len(tokens)
    avg_logprob = total_logprob / total_words
    ppl = 2**(-avg_logprob)
    return ppl

### In the above perplexity function we calculate the perplexity of our trigram model, we generate the log probabilities of each token as a trigram model, and then calculate the perplexity using markov property of the trigram model.

Call the `perplexity` function, and calculate the perplexity of with the file `lang-model.txt` (the training data) with three different $\alpha$'s

- $\alpha=0.001$ (the default value)
- $\alpha=0.1$

And report the perplexity numbers respectively.

### alpha=0.001 (1 point)

TODO

- Please calculate the perplexity with $\alpha=0.001$, and report the perplexity number.

In [29]:
# Code block for alpha=0.001
model =  trigram
ppl = perplexity(model, 'lang-model.txt')
print(f'Perplexity with alpha=0.001: {ppl}')

Perplexity with alpha=0.001: 22.695273062808354


In [30]:
# calculating perplexity with alpha=0.01
trigram_0_01 = TrigramLM()
trigram_0_01.build("lang-model.txt",alpha=0.01)
model = trigram_0_01
ppl = perplexity(model, 'lang-model.txt')
print(f'Perplexity with alpha=0.01: {ppl}')

Perplexity with alpha=0.01: 75.71148596974326


### alpha=0.1 (1 point)

TODO:

- Please calculate the perplexity with $\alpha=0.1$, and report the perplexity number.

In [31]:
# TODO: code block for alpha=0.1
model = trigram_0_1
ppl = perplexity(model, 'lang-model.txt')
print(f'Perplexity with alpha=0.1: {ppl}')

Perplexity with alpha=0.1: 404.63043565758426


### Please Note that the perplexities can vary based on the implementation. Here since we are adding one more start token to the text in training and calculation of the logprob values and are then discarding the start and end token as suggested in the logprob function, the perplexity values could be slightly different from the expected ones. However the perplexity is exhibiting the expected behaviour of increasing as the alpha value increases.

## Generation (3 points)

Use the `generate` function implemented in the previous section for testing.

### Random Sampling as Greedy Decoding (1.5 points)

TODO:

- Please pick a temperature value that random sampling can be considered as greedy decoding for generation.

In [32]:
# TODO:
# Picking a temperature value that makes random sampling approximate greedy decoding

temp = 0.01
text = model.generate(method='random', length=20, hp=temp)
print(f'Generated text with temp={temp}: {text}')

Generated text with temp=0.01: we present an approach to the best of our model is able to achieve this we propose novel method for


TODO:

- Justify your choice of the temperature value in the following text block.

Greedy decoding is a decoding method where we always select the token with the highest probability of appearing next based on our previous text, in this case since it is a trigram model the previous two words. To make the random sampling algorithm approximate greedy decoding we need to set the temperature to a very low value, as this will enhance the probability of the token with the highest probability of appearing next in the softmax distribution. This is because in the softmax function as the temperature approaches zero the softmax function approaches a one-hot encoding. This means that the token with the highest probability will have a much higher probability of being selected compared to the other tokens. This is why we set the temperature to a very low value to approximate greedy decoding. Here we set the temperature to 0.01 to approximate greedy decoding, which is the least possible value without breaking the code and the random weighted selection function. We can verify if this is the case by generating a text with this temperature multiple times and checking if the same text is generated each time since our initial two tokens are the same each time (start tokens). You can also set two custom tokens as the start tokens in the generation function to check if the same text is generated each time.

### Top-$p$ as Greedy Decoding (1.5 points)

TODO:

- Please pick a $p$ value in the top-$p$ algorithm, which will let top-$p$ approximately be a greedy decoding algorithm.

In [33]:
# TODO:

p = 0.000001
text = model.generate(method='top-p', length=20, hp=p)
print(f'Generated text with p={p}: {text}')

Generated text with p=1e-06: we present an approach to the best of our model is able to achieve this we propose novel method for


TODO:

- Justify your choice of the hyper-parameter in the following text block.

Here we use a very small value for hyperparameter p to approximate greedy decoding. In the top-p sampling algorithm, we sort the tokens by probability and then select the tokens with the highest probabilities until the cumulative probability reaches p. By setting p to a very small value we are effectively selecting only the token with the highest probability, which is the same as greedy decoding. This is because the cumulative probability will reach p with only the token with the highest probability. We can verify if this is the case by generating a text with this hyperparameter multiple times and checking if the same text is generated each time since our initial two tokens are the same each time (start tokens). You can also set two custom tokens as the start tokens in the generation function to check if the same text is generated each time. This also can be verified as true because we get the same text output from the random sampling and top-p sampling algorithms with the hyperparameters set to approximate greedy decoding.

# End of HW-3

*Assisted By Github Copilot and ChatGPT

*References:
- https://medium.com/nlplanet/two-minutes-nlp-most-used-decoding-methods-for-language-models-9d44b2375612
- https://spotintelligence.com/2024/08/19/perplexity-in-nlp/